# Finetuning on GERA + LORuGEC

In [1]:
import os, sys
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'pipeline')) else os.path.abspath(os.path.join(_here, os.pardir))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print('project root:', _root)

project root: /home/temari/god please no diploma/restore_punct


## Run config

In [ ]:
from pipeline.env import DATA_DIR, MODEL_DIR
from pipeline.config import RunConfig

LORUGEC_UPSAMPLE = 13
lorugec_files = [f"{DATA_DIR}/train_lorugec.json"] * LORUGEC_UPSAMPLE


cfg = RunConfig(
    name          = "04b_finetune_replay_v3",
    architecture  = "bert+crf",
    loss          = "crf",
    train_files   = [
        f"{DATA_DIR}/train_gera.json",
        *lorugec_files,
    ],
    val_files     = [f"{DATA_DIR}/val_gera.json"],
    replay_files  = [{"path": f"{DATA_DIR}/train_all_rare_punct.json", "n": 10000, "seed": 42}],
    init_from     = f"{MODEL_DIR}/03_crf_transitions_mined_v2",
    num_epochs    = 3,
    learning_rate = 5e-6,
    warmup_ratio  = 0.1,
    crf_init_transitions = False,
    tags          = {"stage": "4-gera-lorugec", "strategy": "finetune+replay-v2"},
)


print(cfg)

RunConfig(name='04b_finetune_replay_v3', architecture='bert+crf', loss='crf', train_files=['/home/temari/god please no diploma/restore_punct/data/train_gera.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', 

## Train

In [3]:
from pipeline.training import train_run
model = train_run(cfg)

[04b_finetune_replay_v3] architecture=bert+crf loss=crf epochs=3 lr=5e-06
[04b_finetune_replay_v3] warm-starting from /home/temari/god please no diploma/restore_punct/models/03_crf_transitions_mined_v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/home/temari/anaconda3/envs/neural/lib/python3.13/site-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:614.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,17.571543,2.192617,0.960575,0.961385,0.960224,0.961385
2,14.796913,2.139436,0.962064,0.963033,0.961836,0.963033
3,15.447899,2.138231,0.962048,0.963033,0.961897,0.963033


Model saved -> /home/temari/god please no diploma/restore_punct/models/04b_finetune_replay_v3


## Benchmark + save results

In [4]:
from pipeline.evaluation import evaluate_and_save
reports = evaluate_and_save(cfg)

for test_name, rep in reports.items():
    wa = rep.get('weighted avg', {})
    ma = rep.get('macro avg', {})
    print(f"{test_name:>14s} | W-F1={wa.get('f1-score', 0):.4f}  M-F1={ma.get('f1-score', 0):.4f}  P={wa.get('precision', 0):.4f}  R={wa.get('recall', 0):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[04b_finetune_replay_v3] evaluating on General_Test (n=569)


Evaluating:   0%|          | 0/143 [00:00<?, ?it/s]

[04b_finetune_replay_v3] evaluating on GERA_Test (n=1259)


Evaluating:   0%|          | 0/315 [00:00<?, ?it/s]

[04b_finetune_replay_v3] evaluating on LORuGEC_Test (n=603)


Evaluating:   0%|          | 0/151 [00:00<?, ?it/s]

Updated /home/temari/god please no diploma/restore_punct/results/models_db.json (entry: 04b_finetune_replay_v3)
Wrote /home/temari/god please no diploma/restore_punct/results/04b_finetune_replay_v3.json
Wrote /home/temari/god please no diploma/restore_punct/results/04b_finetune_replay_v3.xlsx
  General_Test | W-F1=0.9529  M-F1=0.4570  P=0.9520  R=0.9546
     GERA_Test | W-F1=0.9623  M-F1=0.5124  P=0.9625  R=0.9635
  LORuGEC_Test | W-F1=0.9729  M-F1=0.5317  P=0.9730  R=0.9735


## Example run

In [5]:
from pipeline.inference import load_for_inference, restore_punctuation

m, tok = load_for_inference(cfg)
for t in [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
    "почему вы сидите - сказал Вася. А мы просто устали ответила Маша",
    "где был этот ваш будильник когда я спал",
    "почему так красиво березы шумят",
    "что делаешь спросила наташа"

]:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(m, tok, t)}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Input:    Мама мыла раму а папа читал газету
Restored: Мама мыла раму, а папа читал газету.

Input:    Однако мы решили не идти в кино потому что пошел дождь
Restored: Однако мы решили не идти в кино, потому что пошел дождь.

Input:    Он сказал Привет как дела
Restored: Он сказал: " Привет, как дела".

Input:    Я говорю ей Что думаешь А она мне Да ничего отстань уже
Restored: Я говорю ей: " Что думаешь?" А она мне: " Да ничего, отстань уже".

Input:    После многих страданий А А Пушкин всё-таки написал свои стишки
Restored: После многих страданий А. А. Пушкин всё-таки написал свои стишки.

Input:    почему вы сидите - сказал Вася. А мы просто устали ответила Маша
Restored: почему вы сидите? - сказал Вася. - А мы просто устали, - ответила Маша.

Input:    где был этот ваш будильник когда я спал
Restored: где был этот ваш будильник, когда я спал?

Input:    почему так красиво березы шумят
Restored: почему так красиво березы шумят?

Input:    что делаешь спросила наташа
Restored: что де

## Stats

In [6]:
from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()

Rebuilt master table -> /home/temari/god please no diploma/restore_punct/results/master_summary.xlsx
  BERT runs   : 16
  Yandex runs : 1


'/home/temari/god please no diploma/restore_punct/results/master_summary.xlsx'